**Figure 5: Causal Recovery Retains Useful Work.** A controlled effect-DAG workload varies trajectory size, dependency shape, fault position, and the fraction of independent work (3 repeats). (a) AgentTX causal rollback keeps all independent calls as the DAG grows, while temporal and whole-session rollback discard useful work; (b) dependency-aware rollback removes every invalid descendant, unlike the no-dependency ablation; (c) the joint recovery utility (useful work retained $\times$ invalid work removed) remains robust across fault positions; (d) selective reconstruction adds bounded recovery latency. All recovery policies receive identical declared read effects except the explicit dependency-capture ablation.

In [ ]:
# ipython -c "%run plot_causal_retention.ipynb"
# FAST/USENIX line-plot conventions: white panels, boxed top legend, red solid squares = ours.
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from pathlib import Path

STANDARD_WIDTH = 17.8            # USENIX two-column text width, cm

def cm_to_inch(value):
    return value / 2.54

plt.rcParams.update(plt.rcParamsDefault)
matplotlib.rcParams['text.usetex'] = False
plt.rcParams['font.family'] = 'Nimbus Roman'
plt.rcParams['axes.grid'] = False
plt.rcParams['axes.linewidth'] = 0.6
plt.rcParams['xtick.direction'] = 'in'
plt.rcParams['ytick.direction'] = 'in'
plt.rcParams['legend.frameon'] = True
plt.rcParams['legend.edgecolor'] = '0.55'
plt.rcParams['legend.framealpha'] = 1.0
plt.rcParams['legend.fancybox'] = False

STYLES = {
    'causal': dict(color='#c00000', marker='s', linestyle='-', linewidth=1.0, markersize=3.2),
    'temporal': dict(color='#e78129', marker='x', linestyle=':', linewidth=0.9, markersize=3.6, markeredgewidth=0.9),
    'whole_session': dict(color='black', marker='o', linestyle='--', linewidth=0.8, markersize=2.8, markerfacecolor='none'),
    'causal_without_dependencies': dict(color='#4f9fcf', marker='^', linestyle='-.', linewidth=0.9, markersize=3.2, markerfacecolor='none'),
}
LABELS = {
    'causal': 'AgentTX causal (ours)',
    'temporal': 'temporal rollback',
    'whole_session': 'whole-session discard',
    'causal_without_dependencies': 'causal, no dependencies',
}
MODES = list(STYLES)

cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == 'motivation' else cwd
RESULTS = ROOT / 'experiments' / 'results'
FIGDIR = ROOT / 'motivation'
df = pd.read_csv(RESULTS / 'causal_retention.csv')

def sweep_series(sweep, metric, mode):
    rows = df[(df['sweep'] == sweep) & (df['mode'] == mode)].copy()
    rows['x_num'] = pd.to_numeric(rows['x_value'])
    rows = rows.sort_values('x_num')
    return rows['x_num'].to_numpy(), rows[metric].to_numpy(dtype=float)

fig = plt.figure(dpi=300, figsize=(cm_to_inch(STANDARD_WIDTH), cm_to_inch(7.0)))
handles = []

ax = plt.subplot(2, 2, 1)
for mode in MODES:
    x, y = sweep_series('size', 'independent_retention_mean', mode)
    handle, = ax.plot(x, 100 * y, **STYLES[mode], label=LABELS[mode])
    handles.append(handle)
ax.set_ylabel('Useful work retained (%)', fontsize=8)
ax.set_xlabel('DAG size (# calls)\n(a) Independent work', fontsize=7)
ax.set_ylim(-5, 105)

ax = plt.subplot(2, 2, 2)
for mode in MODES:
    x, y = sweep_series('size', 'target_removed_mean', mode)
    ax.plot(x, 100 * y, **STYLES[mode])
ax.set_ylabel('Invalid work removed (%)', fontsize=8)
ax.set_xlabel('DAG size (# calls)\n(b) Recovery completeness', fontsize=7)
ax.set_ylim(-5, 105)

ax = plt.subplot(2, 2, 3)
for mode in MODES:
    rows = df[(df['sweep'] == 'fault_position') & (df['mode'] == mode)].copy()
    rows['x_num'] = pd.to_numeric(rows['x_value']) * 100
    rows = rows.sort_values('x_num')
    utility = rows['independent_retention_mean'].to_numpy(dtype=float) * rows['target_removed_mean'].to_numpy(dtype=float)
    ax.plot(rows['x_num'], 100 * utility, **STYLES[mode])
ax.set_ylabel('Recovery utility (%)', fontsize=8)
ax.set_xlabel('Requested fault position (%)\n(c) Retention x removal', fontsize=7)
ax.set_ylim(-5, 105)

ax = plt.subplot(2, 2, 4)
for mode in MODES:
    x, y = sweep_series('size', 'rollback_ms_p95', mode)
    ax.plot(x, y, **STYLES[mode])
ax.set_ylabel('Rollback p95 (ms)', fontsize=8)
ax.set_xlabel('DAG size (# calls)\n(d) Recovery latency', fontsize=7)

for ax in fig.axes:
    ax.tick_params(axis='both', labelsize=7)

fig.legend(handles=handles, loc='upper center', bbox_to_anchor=(0.5, 1.035), ncol=4,
           fontsize=6.5, columnspacing=0.8, handlelength=1.8, handletextpad=0.35, borderpad=0.3)
plt.tight_layout(pad=0.6, h_pad=1.6, w_pad=1.2, rect=[0.0, 0.0, 1.0, 0.91])
plt.savefig(FIGDIR / 'FIG-Causal-Retention.pdf', bbox_inches='tight', pad_inches=0.02)
plt.savefig(FIGDIR / 'FIG-Causal-Retention.png', dpi=300, bbox_inches='tight', pad_inches=0.02)
plt.show()

size_rows = df[df['sweep'] == 'size'].copy()
size64 = size_rows[pd.to_numeric(size_rows['x_value']) == 64].set_index('mode')
for mode in MODES:
    row = size64.loc[mode]
    print(f"{LABELS[mode]:28s}: retain={row['independent_retention_mean']:.1%}, remove={row['target_removed_mean']:.1%}, rollback_p95={row['rollback_ms_p95']:.1f} ms")
